# Source audit and one-row-per-yield warehouse

In [1]:
import json
from pathlib import Path

report = json.loads(Path("reports/quality-report.json").read_text())
report

{'source_audit': {'yield': {'source_rows': 45499,
   'loaded_rows': 45499,
   'quarantined_rows': 0,
   'sha256': '20bed8fee96e614ea7f254d1140f401c77fd16d08eecfb0db1a0cb490852c5b3',
   'missing_cells': 0,
   'duplicate_key_rows': 0},
  'soil': {'source_rows': 917,
   'loaded_rows': 917,
   'quarantined_rows': 0,
   'sha256': 'd71073640368946d50b59a2822f2d6d0543b5c51e1eeb85854ca79fd02025fe7',
   'missing_cells': 0,
   'duplicate_key_rows': 0},
  'weather': {'source_rows': 577980,
   'loaded_rows': 577980,
   'quarantined_rows': 0,
   'sha256': '7458c1682fea0494c61e812ea1162256344441186f7991b40d8a19bcb7427201',
   'missing_cells': 0,
   'duplicate_key_rows': 0}},
 'coverage': {'yield_rows': 45499,
  'missing_soil_rows': 23980,
  'incomplete_weather_rows': 29444,
  'zero_yield_rows': 501},
 'scope': 'Complete archived county tables; missing joins are coverage warnings; soil units unverified'}

In [2]:
import sqlite3
from contextlib import closing

import pandas as pd

with closing(sqlite3.connect("data/processed/agriculture.sqlite")) as db:
    counts = db.execute("SELECT COUNT(*) FROM yield_coverage").fetchone()[0]
    assert counts == report["coverage"]["yield_rows"] == 45499
    assert not db.execute("PRAGMA foreign_key_check").fetchall()
    coverage = pd.read_sql_query(
        "SELECT year, COUNT(*) AS yield_rows,SUM(weather_dekads=36) AS complete_weather FROM yield_coverage GROUP BY year",
        db,
    )
coverage

,year,yield_rows,complete_weather
0,1994,2149,0
1,1995,2067,0
2,1996,2111,0
3,1997,2077,0
4,1998,2092,0
5,1999,2067,0
6,2000,2047,902
7,2001,1991,900
8,2002,2054,912
9,2003,2025,913


No real source row was quarantined. Missing joins stay visible; soil units and zero-yield semantics require review. Artificial test fixtures do not enter these reports.